In [0]:
%pip install usaddress openpyxl
%restart_python

In [0]:
%run ./utils/usaddress_parse_util

In [0]:
"""Pipeline configuration.

Adjustable: catalog and schema names, volume path, file glob, sheet name,
checkpoint location, roster table.

Fixed: COLUMN_SPEC and BRONZE_STRUCT. Field order in BRONZE_STRUCT defines the
write contract and must match the target table.
"""
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

CATALOG = "bis_prod"
BRONZE_SCHEMA = "bronze_roster"
SILVER_SCHEMA = "silver_roster"

VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/manual_inputs"
CHECKPOINT = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/manual_inputs/_checkpoints/emp_storage_location"

# Strict glob for structural isolation, since manual_inputs is a shared drop zone
FILE_GLOB = "*[Ss]torage*[Ss]pace*.xlsx"
SHEET_NAME = "Spaces"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.emp_storage_location"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.emp_storage_location"

ROSTER_TABLE = None  # set to enable the two email-match QA checks

# source header -> (target column, nullable)
COLUMN_SPEC = {
    "RepName":          ("Rep_Name",             False),
    "Rep Email":        ("Rep_Email",            False),
    "Rep Phone":        ("Rep_Phone",            False),
    "Territory ID":     ("Territory_ID",         False),
    "Emp ID":           ("Emp_ID",               False),
    "Facility Name":    ("Facility_Name",        False),
    "Facility Address": ("Facility_Raw_Address", False),
    "Space Number":     ("Space_Number",         False),
    "Size":             ("Storage_Size",         False),
    "Territory Name":   ("Territory_Name",       False),
    "Director":         ("Region_Emp_Name",      True),
    "Region Name":      ("Region_Name",          True),
}

# Column order here is the write contract. load_batch builds tuples from
# [f.name for f in BRONZE_STRUCT.fields], so the two can never drift.
BRONZE_STRUCT = StructType(
    [StructField(target, StringType(), True) for target, _ in COLUMN_SPEC.values()]
    + [StructField(c, StringType(), True) for c in PARSED_COLUMNS]
    + [
        StructField("File_Date", DateType(), True),
        StructField("File_Name", StringType(), True),
        StructField("Load_Timestamp", TimestampType(), True),
        StructField("_rescued_data", StringType(), True),
    ]
)

In [0]:
import json
from io import BytesIO
from openpyxl import load_workbook

def clean(value):
    """Normalize one cell value to a trimmed string or None.

    Args:
        value: Cell value from openpyxl (str, int, float, datetime or None).

    Returns:
        str | None: Trimmed string, or None if blank or whitespace only.
    """
    if value is None:
        return None
    return str(value).strip() or None


def select_sheet(workbook, file_name):
    """Select the worksheet to read.

    Prefers SHEET_NAME, falling back to the only visible sheet when exactly
    one exists.

    Args:
        workbook (openpyxl.Workbook): Opened workbook.
        file_name (str): Source file name, used in error messages.

    Returns:
        tuple: (worksheet, note) where note is None on the normal path or a
            string describing the fallback.

    Raises:
        ValueError: SHEET_NAME is absent and no unambiguous fallback exists.
    """
    try:
        return workbook[SHEET_NAME], None
    except KeyError:
        visible = [ws.title for ws in workbook.worksheets if ws.sheet_state == "visible"]
        if len(visible) == 1:
            return workbook[visible[0]], (
                f"sheet '{SHEET_NAME}' missing, used sole visible sheet '{visible[0]}'"
            )
        raise ValueError(
            f"Sheet '{SHEET_NAME}' not found in {file_name} and fallback is ambiguous. "
            f"Visible sheets: {visible}"
        )


def classify_header(header):
    """Map a header row onto target column names.

    Matches by name, case and whitespace insensitive. Position is ignored, so
    reordered columns are handled transparently.

    Args:
        header (tuple): First row of the worksheet.

    Returns:
        tuple: (index_by_target, rescued_names, missing).
            index_by_target (dict): Target column -> row index.
            rescued_names (dict): Row index -> header text, for columns not in
                COLUMN_SPEC.
            missing (list): Expected columns absent from the header.
    """
    def key(h):
        return str(h).strip().lower() if h is not None else ""

    seen = {key(h): i for i, h in enumerate(header) if key(h)}
    index_by_target, missing, matched = {}, [], set()

    for source, (target, _) in COLUMN_SPEC.items():
        position = seen.get(key(source))
        if position is None:
            missing.append(target)
        else:
            index_by_target[target] = position
            matched.add(position)
    
    # input-file headers whose column indexes were not matched by COLUMN_SPEC
    rescued_names = {
        i: (clean(header[i]) or f"_c{i}")
        for i in range(len(header))
        if i not in matched
    }
    return index_by_target, rescued_names, missing


def read_workbook(content, file_name, file_date, load_ts):
    """Decode one Excel file into records ready for the target schema.

    Args:
        content (bytes): Raw file bytes.
        file_name (str): Source file name.
        file_date (datetime.date): Value for the File_Date column.
        load_ts (datetime.datetime): Value for the Load_Timestamp column.

    Returns:
        tuple: (records, notes).
            records (list[dict]): One dict per data row, keyed to the target
                schema.
            notes (dict): Keys "sheet" and "rescued_columns".

    Raises:
        ValueError: The worksheet is missing or ambiguous, the worksheet is
            empty, or an expected column is absent from the header. All abort
            the caller without writing.
    """
    workbook = load_workbook(BytesIO(content), read_only=True, data_only=True)
    try:
        sheet, sheet_note = select_sheet(workbook, file_name)
        rows = list(sheet.iter_rows(values_only=True))
    finally:
        workbook.close()

    if not rows:
        raise ValueError(f"Sheet '{SHEET_NAME}' is empty in {file_name}")

    header = rows[0]
    index_by_target, rescued_names, missing = classify_header(header)

    # Expected fields missing from input headers.
    if missing:
        raise ValueError(f"{file_name} missing column(s): {missing}")

    records = []

    for raw_row in rows[1:]:
        values = [clean(v) for v in raw_row]
        # Skips fully empty row.
        if not any(values):
            continue  # phantom trailing row - Excel used-range artifact

        # Build the mapped output record from this row's recognised columns.
        record = {
            target: (values[i] if i < len(values) else None)
            for target, i in index_by_target.items()
        }

        # Preserve non-empty input values from unmapped or headerless columns. _c → (column) i → (index)
        rescued = {}
        for i, value in enumerate(values):
            if value is None:
                continue
            if i in rescued_names:
                rescued[rescued_names[i]] = value
            elif i >= len(header):
                rescued[f"_c{i}"] = value
        
        # Parse the US address, add file metadata and rescued fields, then save the completed record.
        record.update(parse_address(record.get("Facility_Raw_Address")))
        record["File_Date"] = file_date
        record["File_Name"] = file_name
        record["Load_Timestamp"] = load_ts
        record["_rescued_data"] = json.dumps(rescued) if rescued else None
        records.append(record)

    notes = {"sheet": sheet_note, "rescued_columns": list(rescued_names.values())}
    return records, notes

In [0]:
from datetime import datetime

def load_batch(batch_df, batch_id):
    """Decode a micro-batch of files and append the rows to the target table.

    Executes on the driver. File bytes are collected locally and decoded in
    Python, so batch size is bounded by driver memory.

    Args:
        batch_df (DataFrame): Batch with columns path, content and
            modificationTime.
        batch_id (int): Micro-batch identifier, used for logging.

    Returns:
        None: Rows are appended as a side effect.

    Raises:
        ValueError: Propagated from read_workbook. Aborts the batch and leaves
            the stream checkpoint unadvanced, so the file is retried.
    """
    records, notes = [], []
    load_ts = datetime.now()
    print(f"[{batch_id}] batch received: {batch_df.count()} file(s)")
    # Plain Python is used here because each file is binary Excel and must be decoded row-wise.
    # pandas_udf is not suitable for binary Excel decoding; it works best for columnar operations on DataFrames.
    # For this workload, plain Python is the recommended approach.

    for row in batch_df.select("path", "content", "modificationTime").collect():
        file_name = row["path"].rsplit("/", 1)[-1]
        file_date = row["modificationTime"].date()
        file_records, file_notes = read_workbook(row["content"], file_name, file_date, load_ts)
        records.extend(file_records)
        notes.append((file_name, len(file_records), file_notes))

    if not records:
        return

    columns = [f.name for f in BRONZE_STRUCT.fields]
    tuples = [tuple(r.get(c) for c in columns) for r in records]

    (
        spark.createDataFrame(tuples, schema=BRONZE_STRUCT)
        .write.mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    for file_name, count, file_notes in notes:
        print(f"[{batch_id}] {file_name}: {count} rows | {file_notes}")

In [0]:
(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", FILE_GLOB)
    .load(VOLUME_PATH)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .foreachBatch(load_batch)
    .start()
    .awaitTermination()
)

In [0]:
QA_RESULT_SCHEMA = "check_name string, severity string, fail_count int, description string"


def run_qa(checks):
    """Run a list of QA checks and report the results.

    Checks are declarative: each is a tuple, and the framework has no knowledge
    of what any individual check means.

    Nothing raises on failure. Every check reports and the run continues, which
    matches the intent that duplicates and parse failures are surfaced rather
    than treated as fatal.

    Args:
        checks (list[tuple]): (check_name, severity, description, sql).
            sql must return one row with one column holding a failure count.
            Pass sql=None to record the check as SKIPPED without running it.

    Returns:
        DataFrame: One row per check, matching QA_RESULT_SCHEMA.
    """
    rows = []

    for name, severity, description, sql in checks:
        if sql is None:
            rows.append((name, "SKIPPED", 0, description))
            continue
        try:
            count = int(spark.sql(sql).collect()[0][0] or 0)
            rows.append((name, severity, count, description))
        except Exception as e:
            # A broken check must not hide the results of the others
            rows.append((name, "CHECK_FAILED", -1, f"{description} | {e}"))

    failures = [r for r in rows if r[2] != 0]
    if failures:
        for name, severity, count, description in failures:
            print(f"[{severity}] {name} = {count} — {description}")
    else:
        print(f"All {len(rows)} checks passed.")

    return spark.createDataFrame(rows, QA_RESULT_SCHEMA)

In [0]:
LATEST_FILE = f"(SELECT MAX(File_Date) FROM {BRONZE_TABLE})"

bronze_checks = [
    (
        "email_matches_roster",
        "ERROR",
        "Rep_Email does not match the roster, or Emp_ID is absent from it",
        None if ROSTER_TABLE is None else
        f"""SELECT COUNT(*) FROM {BRONZE_TABLE} b
            LEFT JOIN {ROSTER_TABLE} r ON b.Emp_ID = r.Emp_ID
            WHERE b.File_Date = {LATEST_FILE}
              AND (r.Emp_ID IS NULL OR lower(b.Rep_Email) <> lower(r.Rep_Email))""",
    ),
    (
        "one_location_per_emp_per_file",
        "WARNING",
        "Emp_ID appears more than once in the same file; Silver keeps the most recently loaded row",
        f"""SELECT COUNT(*) FROM (
              SELECT Emp_ID, File_Name FROM {BRONZE_TABLE}
              WHERE File_Date = {LATEST_FILE}
              GROUP BY Emp_ID, File_Name HAVING COUNT(*) > 1)""",
    ),
    (
        "rescued_data_present",
        "WARNING",
        "Source file contained a new, renamed or extra column",
        f"SELECT COUNT(*) FROM {BRONZE_TABLE} "
        f"WHERE File_Date = {LATEST_FILE} AND _rescued_data IS NOT NULL",
    ),
]

display(run_qa(bronze_checks))

In [0]:
%skip
%sql
REFRESH MATERIALIZED VIEW bis_prod.silver_roster.emp_storage_location;

In [0]:
%skip
silver_checks = [
    (
        "address_not_parsed",
        "WARNING",
        "Address_1 is blank or Parsing_Error is populated",
        f"SELECT COUNT(*) FROM {SILVER_TABLE} "
        f"WHERE Address_1 IS NULL OR Parsing_Error IS NOT NULL",
    ),
]

display(run_qa(silver_checks))